In [ ]:
import requests
import pandas as pd
import json
import time
import re
from pathlib import Path
import os
os.chdir(r"C:\Users\Hp\Downloads\Project 2026 DS")

BASE=Path("openactive_providers")
HEADERS={"User-Agent": "LondonSport-MastersProject/1.0 (contact: ah25315@bristol.ac.uk)","Accept": "application/json",}
REQUEST_TIMEOUT=30
SLEEP_BETWEEN_PAGES=0.3

In [ ]:
def first_pref_label(items):
    return items[0].get("prefLabel") if items and isinstance(items[0],dict) else None

def first_price(offers):
    return offers[0].get("price") if offers and isinstance(offers[0],dict) else None

def join_facilityuse_slot(provider_folder,provider_group,source_operator):
    fu_rows=[]
    with open(BASE / provider_folder / "FacilityUse.json") as f:
        fu_items=json.load(f)
    for item in fu_items:
        if not isinstance(item,dict) or item.get("state") == "deleted":
            continue
        d=item.get("data",{}) or {}
        loc=d.get("location",{}) or {}
        if not isinstance(loc,dict):
            loc={}
        address=loc.get("address",{}) or {}
        geo=loc.get("geo",{}) or {}
        activity_type=first_pref_label(d.get("activity",[]) or [])
        if not activity_type:
            activity_type=first_pref_label(d.get("facilityType",[]) or [])
        activity_raw=d.get("category",[])
        activity_raw=activity_raw[0] if isinstance(activity_raw,list) and activity_raw else None
        fu_rows.append({"facility_use_id": item.get("id"),
            "name": d.get("name"),"activity_type": activity_type,
            "activity_raw": activity_raw,"location_name": loc.get("name"),
            "postcode": address.get("postalCode"),"latitude": geo.get("latitude"),
            "longitude": geo.get("longitude"),"url": d.get("url"),})
    fu_df=pd.DataFrame(fu_rows)

    slot_rows=[]
    with open(BASE / provider_folder / "Slot.json") as f:
        slot_items=json.load(f)
    for item in slot_items:
        if not isinstance(item,dict) or item.get("state") == "deleted":
            continue
        d=item.get("data",{}) or {}
        fu_url=d.get("facilityUse","")
        fu_id=fu_url.split("/facility-uses/")[-1] if "/facility-uses/" in fu_url else None
        price_gbp=first_price(d.get("offers",[]) or [])
        start_date=d.get("startDate","")
        end_date=d.get("endDate","")
        slot_rows.append({"facility_use_id": fu_id,"price_gbp": price_gbp,"is_free": (price_gbp == 0) if price_gbp is not None else None, "start_time": start_date.split("T")[1][:5] if start_date and "T" in start_date else None, "end_time": end_date.split("T")[1][:5] if end_date and "T" in end_date else None,})
    slots_df=pd.DataFrame(slot_rows)
    merged=slots_df.merge(fu_df,on="facility_use_id",how="left")
    merged["provider_group"]=provider_group
    merged["source_operator"]=source_operator
    merged["session_count"]=1
    merged["is_online"]=False
    merged["days_of_week"]=None

    return merged[["provider_group","source_operator","name","activity_type","activity_raw","is_free","price_gbp","location_name","postcode","latitude","longitude","start_time","end_time","days_of_week","url","session_count","is_online"]]

def join_sessionseries(provider_folder,provider_group):
    with open(BASE / provider_folder / "SessionSeries.json") as f:
        items=json.load(f)
    rows=[]
    for item in items:
        if not isinstance(item,dict) or item.get("state") == "deleted":
            continue
        d=item.get("data",{}) or {}
        if not isinstance(d,dict):
            continue
        loc=d.get("location",{}) or {}
        if not isinstance(loc,dict):
            loc={}
        address=loc.get("address",{}) or {}
        geo=loc.get("geo",{}) or {}
        super_event=d.get("superEvent",{}) or {}
        if not isinstance(super_event,dict):
            super_event={}
        organizer=super_event.get("organizer",{}) or {}
        if not isinstance(organizer,dict):
            organizer={}
        activity_type=first_pref_label(super_event.get("activity",[]) or [])
        activity_raw=d.get("category",[])
        activity_raw=activity_raw[0] if isinstance(activity_raw,list) and activity_raw else None
        schedules=d.get("eventSchedule",[]) or []
        sched=schedules[0] if schedules and isinstance(schedules[0],dict) else {}
        days_raw=sched.get("byDay",[]) or []
        days_of_week=", ".join(u.replace("https://schema.org/","") for u in days_raw if isinstance(u,str))
        price_gbp=first_price(d.get("offers",[]) or [])

        rows.append({"provider_group": provider_group,"source_operator": organizer.get("name"),
            "name": d.get("name"),"activity_type": activity_type,"activity_raw": activity_raw,
            "is_free": (price_gbp == 0) if price_gbp is not None else None,
            "price_gbp": price_gbp,"location_name": loc.get("name"),
            "postcode": address.get("postalCode"),"latitude": geo.get("latitude"),
            "longitude": geo.get("longitude"),"start_time": sched.get("startTime"),"end_time": sched.get("endTime"),
            "days_of_week": days_of_week if days_of_week else None,"url": d.get("url"),"session_count": 1,
            "is_online": False,})
    return pd.DataFrame(rows)

dfs=[]
dfs.append(join_facilityuse_slot("Ealing_Council","Ealing Council","London Borough of Ealing"))
dfs.append(join_facilityuse_slot("Haringey_Council","Haringey Council","London Borough of Haringey"))
dfs.append(join_facilityuse_slot("WalthamForest_Council","Waltham Forest Council","London Borough of Waltham Forest"))
dfs.append(join_facilityuse_slot("Southwark_Council_bookteq","Southwark Council","London Borough of Southwark"))
dfs.append(join_sessionseries("TowerHamlets_BeWell","Be Well - Tower Hamlets"))
dfs.append(join_sessionseries("Southwark_Council","Southwark Council"))

combined=pd.concat(dfs,ignore_index=True)
combined.to_csv("otherproviders_combined.csv",index=False)
print(f"Done — {len(combined)} rows saved")
print(combined.groupby("provider_group").size())

In [ ]:
ACTIVITY_KEYWORD_MAP=[(["swim","aqua","diving","kayak","paddleboard","watersport"],"Swimming"),
    (["yoga","pilates","tai chi","meditation","barre"],"Yoga, Pilates & Studio"),
    (["badminton","squash","tennis","table tennis","pickleball","fives"],"Racquet Sports"),
    (["football","netball","basketball","cricket","rugby","volleyball","rounders","boccia","multisport","a-side","pitch","softball","baseball"],"Team Sports"),
    (["boxing","martial","karate","judo","boxercise","boxfit"],"Martial Arts"),
    (["bootcamp","outdoor","climb","bouldering","trampolin"],"Bootcamp & Outdoor Fitness"),
    (["run","athletic","walking","track"],"Walking / Running"),
    (["cycl","spin","rpm"],"Cycling"),
    (["personal training","1-2-1","pt "],"Personal Training"),
    (["dance","salsa","zumba","latin","ballroom","line danc"],"Dance"),
    (["gym","fitness","hiit","circuit","conditioning","strength","bodypump","bodycombat","bodyattack","bodybalance","bodystep","core","abs","legs bums","stretch","cx worx","grit","kettlebell","weights","toning","lifting","calisthenics","kickstart","hyrox","battle blast","pull up","functional","double impact","induction"],"Gym / Fitness"),
    (["class","aerobic","exercise","workout"],"Group Exercise"),
    (["ice skat"],"Ice Skating & Wheeled Sports"),
    (["sauna","spa","soft play","good boost land","sound bath","focus","activate"],"Wellness & Facility Access"),]

def _normalize(text):
    return re.sub(r'[^a-z]','',str(text).lower())

def map_activity_type(activity_raw):
    if not activity_raw:
        return "Other / Unspecified"
    text=_normalize(activity_raw)
    for keywords,category in ACTIVITY_KEYWORD_MAP:
        if any(_normalize(kw) in text for kw in keywords if _normalize(kw)):
            return category
    return "Other / Unspecified"


In [ ]:
leisurecloud_providers=["Southwark Council","Be Well - Tower Hamlets"]
mask_lc=combined["provider_group"].isin(leisurecloud_providers)
combined.loc[mask_lc,"activity_type"]=combined.loc[mask_lc,"name"].apply(map_activity_type)
print(combined.groupby("provider_group")["activity_type"].value_counts())

In [ ]:
BOOKTEQ_NAME_TO_ACTIVITY={"11-a-side 3G Pitch": "Team Sports","7-a-side 3G Pitch 1": "Team Sports",
    "7-a-side 3G Pitch 2": "Team Sports","7-a-side 3G Pitch 3": "Team Sports","11-a-side Football": "Team Sports","5-a-side Football": "Team Sports",
    "7 a-side Football": "Team Sports","7-a-side Football": "Team Sports",
    "7-a-side Football 1": "Team Sports","7-a-side Football 2": "Team Sports","7-a-side Football 3": "Team Sports","7-a-side Pitch 1": "Team Sports",
    "7-a-side Pitch 2": "Team Sports","9-aside Astroturf": "Team Sports","Badminton Courts": "Racquet Sports","Basketball Court": "Team Sports","Cricket Nets": "Team Sports","Futsal Pitch 1": "Team Sports",
    "Futsal Pitch 2": "Team Sports","Grass Field": "Team Sports",
    "Hockey Full Pitch": "Team Sports","Hockey Half Pitch 1": "Team Sports","Hockey Half Pitch 2": "Team Sports","Indoor Cricket Net 1": "Team Sports",
    "Indoor Cricket Net 2": "Team Sports","Indoor Cricket Net 3": "Team Sports","Indoor Cricket Net 4": "Team Sports","Indoor Cricket Nets (Whole Hall)": "Team Sports",
    "Netball Court": "Team Sports","Outdoor Net 1": "Team Sports","Outdoor Net 2": "Team Sports","Table Tennis": "Racquet Sports",
    "Volleyball": "Team Sports","Volleyball Court 1": "Team Sports","Volleyball Court 2": "Team Sports","Volleyball Court 3": "Team Sports",
    "11-a-side - Non Affiliated Clubs (Haringey Council)": "Team Sports","11-a-side Affiliated Clubs (Haringey Council) ": "Team Sports",
    "9-aside - Affiliated Clubs (Haringey Council)": "Team Sports","Aussie Rules": "Team Sports","Cricket Booking 3, 5 and 6 hrs ": "Team Sports",
    "Gaelic Football Pitch": "Team Sports","Rugby Pitch": "Team Sports","Cricket Grass Pitch": "Team Sports",}

bookteq_providers=["Waltham Forest Council","Haringey Council","Ealing Council"]
mask=combined["provider_group"].isin(bookteq_providers)
combined.loc[mask,"activity_type"]=(combined.loc[mask,"name"].map(BOOKTEQ_NAME_TO_ACTIVITY).fillna("Other / Unspecified"))
combined.loc[mask & combined["activity_raw"].isna(),"activity_raw"]=combined.loc[mask,"name"]
print(combined[combined["provider_group"].isin(bookteq_providers)]["activity_type"].value_counts())

In [ ]:
LONDON_POSTCODES=("E","EC","N","NW","SE","SW","W","WC","BR","CR","DA","EN","HA","IG","KT","RM","SM","TW","UB","WD")

def is_london_postcode(pc):
    if not isinstance(pc,str):
        return False
    pc=pc.strip().upper()
    return any(pc.startswith(prefix) for prefix in LONDON_POSTCODES)
combined_london=combined[combined["postcode"].apply(is_london_postcode)].copy()
combined_london.to_csv("otherproviders_london.csv",index=False)
print(f"London filter: {len(combined_london)} rows from {len(combined)} total")
print(combined_london.groupby("provider_group").size())

In [ ]:
df=pd.read_csv("otherproviders_london.csv")
deduped=(df.groupby(["provider_group","source_operator","name","activity_type","activity_raw","location_name","postcode","latitude","longitude"],dropna=False).agg(session_count=("session_count","sum"),price_gbp=("price_gbp","min"),is_free=("is_free","any"), start_time=("start_time","first"),end_time=("end_time","first"),days_of_week=("days_of_week","first"),url=("url","first"),is_online=("is_online","first"),).reset_index())
deduped.to_csv("otherproviders_london.csv",index=False)
print(f"Rows after dedup: {len(deduped)}")
print(deduped.groupby("provider_group").size())

In [ ]:
df=pd.read_csv("otherproviders_london.csv")
print(df.groupby("provider_group")["activity_type"].value_counts())

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

df=pd.read_csv("otherproviders_london.csv")
all_las=gpd.read_file("boundaryfile.geojson")
LONDON_BOROUGHS=["Barking and Dagenham","Barnet","Bexley","Brent","Bromley","Camden",
    "City of London","Croydon","Ealing","Enfield","Greenwich","Hackney","Hammersmith and Fulham","Haringey","Harrow","Havering","Hillingdon",
    "Hounslow","Islington","Kensington and Chelsea","Kingston upon Thames",
    "Lambeth","Lewisham","Merton","Newham","Redbridge","Richmond upon Thames","Southwark","Sutton","Tower Hamlets",  "Waltham Forest","Wandsworth","Westminster"]
boroughs=all_las[all_las["LAD24NM"].isin(LONDON_BOROUGHS)].copy()

df_geo=df.dropna(subset=["latitude","longitude"]).copy()
gdf=gpd.GeoDataFrame(df_geo,geometry=[Point(xy) for xy in zip(df_geo.longitude,df_geo.latitude)],crs="EPSG:4326")
gdf=gdf.to_crs(boroughs.crs)
joined=gpd.sjoin(gdf,boroughs[["LAD24NM","geometry"]],how="left",predicate="within")
joined=joined.drop(columns=["geometry","index_right"]).rename(columns={"LAD24NM": "borough"})
print(f"Borough matched: {joined['borough'].notna().sum()} / {len(joined)}")

df=joined
PROVIDER_PREFIX={"Ealing Council": "EAL", "Haringey Council": "HAR", "Waltham Forest Council": "WF", "Southwark Council": "SWK", "Be Well - Tower Hamlets": "TH",}
df=df.reset_index(drop=True)
df["session_id"]=df.apply(lambda row: f"{PROVIDER_PREFIX.get(row['provider_group'], 'OTH')}_{row.name}",axis=1)

SCHEMA_COLUMNS=["session_id","provider_group","source_operator","name","activity_type","activity_raw","is_free","price_gbp","location_name","postcode","borough","latitude","longitude","start_time","end_time","days_of_week","session_count","is_online","url"]
df=df[SCHEMA_COLUMNS]

df.to_csv("otherproviders_london.csv",index=False)
print("Saved. Shape:",df.shape)
print(df.columns.tolist())